# Morphosyntax-Only Parsing: Czech vs. English

## Goal
Compare three input settings for dependency parsing:

1. lexicalized (word identity only)
2. morphosyntax-only (gold UPOS and gold UFeats)
3. mixed (lexical and morphosyntax)

## Datasets
UD Czech-PDT and UD English-EWT, from Universal Dependencies.

## Evaluation
I choose the best epoch on the development set and report final MST-decoded UAS/LAS on a held-out test subset.

I also inspect label confusion patterns and accuracy by dependency length.

## Implementation
I wrote the data reading, vocab building, batching, the graph-based parser model, PyTorch training loop, greedy decoding, MST decoding, and evaluation code myself.

As for off-the-shelf libraries, I used them for CoNLL-U reading (`conllu`) and MST graph utilities (`networkx`).


In [1]:
# fix random seeds for reproducibility
import copy
import random
import numpy as np
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print(f"Seed: {seed}")

Seed: 42


## Section 1: Data Loading
Load UD Czech-PDT and UD English-EWT, then prepare small train/dev/test subsets so the notebook runs in a reasonable time.


In [2]:
try:
    import conllu
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "conllu"])

In [3]:
# clone the two UD treebanks into local data folder
from pathlib import Path
import subprocess

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

repo_urls = {
    "UD_Czech-PDT": "https://github.com/UniversalDependencies/UD_Czech-PDT.git",
    "UD_English-EWT": "https://github.com/UniversalDependencies/UD_English-EWT.git",
}

for repo_name, repo_url in repo_urls.items():
    repo_path = data_dir / repo_name
    if not repo_path.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", repo_url, str(repo_path)],
            check=True,
        )
    print(f"{repo_name}: ready")

UD_Czech-PDT: ready
UD_English-EWT: ready


In [4]:
# locate the CoNLL-U files for English and Czech and split
# Czech-PDT training data is split across multiple shards
czech_repo_path = data_dir / "UD_Czech-PDT"
english_repo_path = data_dir / "UD_English-EWT"

czech_train_paths = sorted(czech_repo_path.glob("cs_pdtc-ud-train-*.conllu"))
czech_dev_path = czech_repo_path / "cs_pdtc-ud-dev.conllu"
czech_test_path = czech_repo_path / "cs_pdtc-ud-test.conllu"

english_train_path = english_repo_path / "en_ewt-ud-train.conllu"
english_dev_path = english_repo_path / "en_ewt-ud-dev.conllu"
english_test_path = english_repo_path / "en_ewt-ud-test.conllu"

print(f"Czech train shards: {len(czech_train_paths)}")
print(f"Czech dev exists: {czech_dev_path.exists()}")
print(f"Czech test exists: {czech_test_path.exists()}")
print(f"English train exists: {english_train_path.exists()}")
print(f"English dev exists: {english_dev_path.exists()}")
print(f"English test exists: {english_test_path.exists()}")

Czech train shards: 12
Czech dev exists: True
Czech test exists: True
English train exists: True
English dev exists: True
English test exists: True


In [5]:
from conllu import parse_incr

def normalize_feats(feats):
    """Return morphological features as a plain dict (empty dict if None)."""
    return dict(feats) if feats is not None else {}

def read_ud_file(conllu_path, sentence_limit=None):
    """Read one CoNLL-U file and return a list of sentences.
    Each sentence is a list of token dicts with form, lemma, upos, feats, head, deprel.
    Multi-word tokens (non-integer ids) are skipped.
    """
    sentence_list = []

    with open(conllu_path, "r") as conllu_file:
        for token_list in parse_incr(conllu_file):
            token_records = []

            for token in token_list:
                # skip multi-word tokens like "1-2"
                if not isinstance(token["id"], int):
                    continue

                token_records.append(
                    {
                        "form": token["form"],
                        "lemma": token["lemma"],
                        "upos": token["upos"],
                        "feats": normalize_feats(token["feats"]),
                        "head": token["head"],
                        "deprel": token["deprel"],
                    }
                )

            if token_records:
                sentence_list.append(token_records)

            if sentence_limit is not None and len(sentence_list) >= sentence_limit:
                break

    return sentence_list

def read_ud_files(conllu_paths, sentence_limit=None):
    """Read multiple CoNLL-U files in order, respecting a global sentence limit."""
    sentence_list = []

    for conllu_path in conllu_paths:
        remaining_limit = None
        if sentence_limit is not None:
            remaining_limit = sentence_limit - len(sentence_list)
            if remaining_limit <= 0:
                break

        file_sentences = read_ud_file(conllu_path, sentence_limit=remaining_limit)
        sentence_list.extend(file_sentences)

    return sentence_list

In [6]:
# load the small subsets to keep training time reasonable
czech_train_small = read_ud_files(czech_train_paths, sentence_limit=2000)
english_train_small = read_ud_file(english_train_path, sentence_limit=2000)

czech_dev_small = read_ud_file(czech_dev_path, sentence_limit=500)
english_dev_small = read_ud_file(english_dev_path, sentence_limit=500)

czech_test_small = read_ud_file(czech_test_path, sentence_limit=500)
english_test_small = read_ud_file(english_test_path, sentence_limit=500)

print("Loaded Czech train sentences:", len(czech_train_small))
print("Loaded Czech dev sentences:", len(czech_dev_small))
print("Loaded Czech test sentences:", len(czech_test_small))
print("Loaded English train sentences:", len(english_train_small))
print("Loaded English dev sentences:", len(english_dev_small))
print("Loaded English test sentences:", len(english_test_small))

Loaded Czech train sentences: 2000
Loaded Czech dev sentences: 500
Loaded Czech test sentences: 500
Loaded English train sentences: 2000
Loaded English dev sentences: 500
Loaded English test sentences: 500


In [7]:
def basic_stats(sentence_list):
    """Return sentence count, token count, and average sentence length."""
    sentence_count = len(sentence_list)
    token_count = sum(len(sentence) for sentence in sentence_list)
    average_sentence_length = token_count / sentence_count
    return sentence_count, token_count, average_sentence_length

# print a qucik overview of each split
dataset_summaries = [
    ("Czech train", czech_train_small),
    ("Czech dev", czech_dev_small),
    ("Czech test", czech_test_small),
    ("English train", english_train_small),
    ("English dev", english_dev_small),
    ("English test", english_test_small),
]

for dataset_name, sentence_list in dataset_summaries:
    print(f"{dataset_name:12s}:", basic_stats(sentence_list))

print("Sample Czech tokens:", czech_train_small[0][:5])
print("Sample English tokens:", english_train_small[0][:5])

Czech train : (2000, 30891, 15.4455)
Czech dev   : (500, 6552, 13.104)
Czech test  : (500, 6862, 13.724)
English train: (2000, 39802, 19.901)
English dev : (500, 7621, 15.242)
English test: (500, 7275, 14.55)
Sample Czech tokens: [{'form': 'Třikrát', 'lemma': 'třikrát', 'upos': 'ADV', 'feats': {'NumType': 'Mult'}, 'head': 2, 'deprel': 'obl'}, {'form': 'rychlejší', 'lemma': 'rychlý', 'upos': 'ADJ', 'feats': {'Case': 'Nom', 'Degree': 'Cmp', 'Gender': 'Fem', 'Number': 'Sing', 'Polarity': 'Pos'}, 'head': 0, 'deprel': 'root'}, {'form': 'než', 'lemma': 'než', 'upos': 'SCONJ', 'feats': {}, 'head': 4, 'deprel': 'mark'}, {'form': 'slovo', 'lemma': 'slovo', 'upos': 'NOUN', 'feats': {'Case': 'Nom', 'Gender': 'Neut', 'Number': 'Sing'}, 'head': 2, 'deprel': 'advcl'}]
Sample English tokens: [{'form': 'Al', 'lemma': 'Al', 'upos': 'PROPN', 'feats': {'Number': 'Sing'}, 'head': 0, 'deprel': 'root'}, {'form': '-', 'lemma': '-', 'upos': 'PUNCT', 'feats': {}, 'head': 3, 'deprel': 'punct'}, {'form': 'Zaman'

### Brief Discussion of Section 1

The two datasets havwe been loaded and kept separate train, dev, and test subsets for English and Czech, because model selection should happen on development data, while the final comparison should stay on held-out test data.

The subsets are still small so the workload remains reasonable.

At this point, the raw CoNLL-U data is ready to be converted into parser-friendly numerical examples.

## Section 2: Sentence Encoding

In this section, the token-level UD records are converted into numerical inputs and gold targets for the parser.

In [8]:
from collections import Counter

def feats_to_text(feats):
    """Convert a feature dict into one sorted string, e.g. 'Case=Nom|Number=Sing'.
    Returns '[NO_FEATS]' when the dict is empty.
    """
    if not feats:
        return "[NO_FEATS]"
    return "|".join(f"{feature_name}={feature_value}" for feature_name, feature_value in sorted(feats.items()))

def build_vocabulary(sentence_list, field_name, min_count=1):
    """Build a value-to-id mapping for one field (form, upos, feats, or deprel).
    [PAD] and [UNK] are always at index 0 and 1.
    """
    value_counter = Counter()

    for sentence in sentence_list:
        for token in sentence:
            value = token[field_name]
            if field_name == "feats":
                value = feats_to_text(value)
            value_counter[value] += 1

    id_to_value = ["[PAD]", "[UNK]"]
    for value, count in value_counter.items():
        if count >= min_count:
            id_to_value.append(value)

    value_to_id = {value: index for index, value in enumerate(id_to_value)}
    return value_to_id, id_to_value

In [9]:
# build actual vocabularies
# words with count < 2 are mapped to [UNK] to reduce vocabulary size
word_to_id, id_to_word = build_vocabulary(czech_train_small + english_train_small, "form", min_count=2)
upos_to_id, id_to_upos = build_vocabulary(czech_train_small + english_train_small, "upos")
feats_to_id, id_to_feats = build_vocabulary(czech_train_small + english_train_small, "feats")
label_to_id, id_to_label = build_vocabulary(czech_train_small + english_train_small, "deprel")

print("Word vocab size:", len(word_to_id))
print("UPOS vocab size:", len(upos_to_id))
print("Feats vocab size:", len(feats_to_id))
print("Label vocab size:", len(label_to_id))
print("First 10 labels:", id_to_label[:10])

Word vocab size: 6301
UPOS vocab size: 19
Feats vocab size: 933
Label vocab size: 61
First 10 labels: ['[PAD]', '[UNK]', 'obl', 'root', 'mark', 'advcl', 'obl:arg', 'advmod:emph', 'amod', 'nsubj']


In [10]:
def add_special_value(value_to_id, id_to_value, special_value):
    """Add one special symbol to a vocabulary if it is missing."""
    if special_value not in value_to_id:
        value_to_id[special_value] = len(id_to_value)
        id_to_value.append(special_value)

# each sentence will start with a synthetic ROOT token
# it needs its own embedding id in each vocabulary
add_special_value(word_to_id, id_to_word, "[ROOT]")
add_special_value(upos_to_id, id_to_upos, "[ROOT_UPOS]")
add_special_value(feats_to_id, id_to_feats, "[ROOT_FEATS]")

root_word_id = word_to_id["[ROOT]"]
root_upos_id = upos_to_id["[ROOT_UPOS]"]
root_feats_id = feats_to_id["[ROOT_FEATS]"]

print("ROOT word id:", root_word_id)
print("ROOT upos id:", root_upos_id)
print("ROOT feats id:", root_feats_id)

ROOT word id: 6301
ROOT upos id: 19
ROOT feats id: 933


In [11]:
def lookup_id(value, value_to_id):
    """Map one value to its id, or use [UNK] if unseen."""
    return value_to_id.get(value, value_to_id["[UNK]"])

def sentence_to_example(sentence):
    """Convert one sentence into aligned parser inputs and gold targets.
    Position 0 is the synthetic ROOT; real tokens start at position 1.
    Head index -1 means 'ignore this position' (used for ROOT and padding).
    """
    token_text_list = ["[ROOT]"]
    word_id_list = [root_word_id]
    upos_id_list = [root_upos_id]
    feats_id_list = [root_feats_id]
    head_index_list = [-1]
    label_id_list = [-1]

    for token in sentence:
        feats_text = feats_to_text(token["feats"])

        token_text_list.append(token["form"])
        word_id_list.append(lookup_id(token["form"], word_to_id))
        upos_id_list.append(lookup_id(token["upos"], upos_to_id))
        feats_id_list.append(lookup_id(feats_text, feats_to_id))
        head_index_list.append(token["head"])
        label_id_list.append(lookup_id(token["deprel"], label_to_id))

    return {
        "token_text_list": token_text_list,
        "word_id_list": word_id_list,
        "upos_id_list": upos_id_list,
        "feats_id_list": feats_id_list,
        "head_index_list": head_index_list,
        "label_id_list": label_id_list,
    }

In [12]:
# sanity check on one encoded Czech sentence
czech_example = sentence_to_example(czech_train_small[0])

print("Tokens:", czech_example["token_text_list"][:6])
print("Word ids:", czech_example["word_id_list"][:6])
print("UPOS ids:", czech_example["upos_id_list"][:6])
print("Head indices:", czech_example["head_index_list"][:6])

label_preview = []
for label_id in czech_example["label_id_list"][:6]:
    if label_id == -1:
        label_preview.append("[IGNORE]")
    else:
        label_preview.append(id_to_label[label_id])

print("Label names:", label_preview)
print("Sentence length with ROOT:", len(czech_example["token_text_list"]))

Tokens: ['[ROOT]', 'Třikrát', 'rychlejší', 'než', 'slovo']
Word ids: [6301, 1, 2, 3, 4]
UPOS ids: [19, 2, 3, 4, 5]
Head indices: [-1, 2, 0, 4, 2]
Label names: ['[IGNORE]', 'obl', 'root', 'mark', 'advcl']
Sentence length with ROOT: 5


In [13]:
def convert_sentences_to_examples(sentence_list):
    """Convert a list of tokenized sentences into parser examples."""
    example_list = []

    for sentence in sentence_list:
        example = sentence_to_example(sentence)
        example_list.append(example)

    return example_list

In [14]:
# convert all splits into numerical examples
czech_train_examples = convert_sentences_to_examples(czech_train_small)
czech_dev_examples = convert_sentences_to_examples(czech_dev_small)
czech_test_examples = convert_sentences_to_examples(czech_test_small)

english_train_examples = convert_sentences_to_examples(english_train_small)
english_dev_examples = convert_sentences_to_examples(english_dev_small)
english_test_examples = convert_sentences_to_examples(english_test_small)

print("Czech train examples:", len(czech_train_examples))
print("Czech dev examples:", len(czech_dev_examples))
print("Czech test examples:", len(czech_test_examples))
print("English train examples:", len(english_train_examples))
print("English dev examples:", len(english_dev_examples))
print("English test examples:", len(english_test_examples))

Czech train examples: 2000
Czech dev examples: 500
Czech test examples: 500
English train examples: 2000
English dev examples: 500
English test examples: 500


In [15]:
def example_length(example):
    """Return the number of positions in one encoded sentence."""
    return len(example["token_text_list"])

print("Average Czech train example length:", np.mean([example_length(x) for x in czech_train_examples]))
print("Average Czech test example length:", np.mean([example_length(x) for x in czech_test_examples]))
print("Average English train example length:", np.mean([example_length(x) for x in english_train_examples]))
print("Average English test example length:", np.mean([example_length(x) for x in english_test_examples]))

Average Czech train example length: 16.4455
Average Czech test example length: 14.724
Average English train example length: 20.901
Average English test example length: 15.55


### Brief Discussion of Section 2A

The sentence encoding step worked correctly. Each sentence starts with a synthetic ROOT token, and the same aligned representation stores word ids, POS ids, feature ids, gold heads, and gold labels in one place.

All six splits (train/dev/test for both languages) are now converted into numerical examples, with Czech averaging around 16 tokens per sentence and English around 20.

The shared vocabulary across both languages ensures that the three parser settings (lexicalized, morphosyntax-only, mixed) will be compared fairly.

This encoding is now ready to be padded into mini-batches for the parser.

## Section 2B: Padding and Batch Masks

Sentences have different lengths, so I pad shorter ones inside each mini-batch and track real tokens with a mask.


In [16]:
import torch

pad_word_id = word_to_id["[PAD]"]
pad_upos_id = upos_to_id["[PAD]"]
pad_feats_id = feats_to_id["[PAD]"]
ignore_index = -1

def pad_example_list(example_list):
    """Pad a list of encoded sentences to the same length and return tensors.
    Padding uses [PAD] ids for inputs, -1 for gold targets, and False in the mask.
    """
    max_length = max(len(example["token_text_list"]) for example in example_list)

    batch_word_ids = []
    batch_upos_ids = []
    batch_feats_ids = []
    batch_head_indices = []
    batch_label_ids = []
    batch_mask = []

    for example in example_list:
        current_length = len(example["token_text_list"])
        padding_length = max_length - current_length

        batch_word_ids.append(
            example["word_id_list"] + [pad_word_id] * padding_length
        )
        batch_upos_ids.append(
            example["upos_id_list"] + [pad_upos_id] * padding_length
        )
        batch_feats_ids.append(
            example["feats_id_list"] + [pad_feats_id] * padding_length
        )
        batch_head_indices.append(
            example["head_index_list"] + [ignore_index] * padding_length
        )
        batch_label_ids.append(
            example["label_id_list"] + [ignore_index] * padding_length
        )
        batch_mask.append(
            [1] * current_length + [0] * padding_length
        )

    return {
        "word_ids": torch.tensor(batch_word_ids, dtype=torch.long),
        "upos_ids": torch.tensor(batch_upos_ids, dtype=torch.long),
        "feats_ids": torch.tensor(batch_feats_ids, dtype=torch.long),
        "head_indices": torch.tensor(batch_head_indices, dtype=torch.long),
        "label_ids": torch.tensor(batch_label_ids, dtype=torch.long),
        "mask": torch.tensor(batch_mask, dtype=torch.bool),
    }

In [17]:
# toy batch shape check
toy_batch = pad_example_list(czech_train_examples[:3])

print("word_ids shape:", toy_batch["word_ids"].shape)
print("upos_ids shape:", toy_batch["upos_ids"].shape)
print("head_indices shape:", toy_batch["head_indices"].shape)
print("mask shape:", toy_batch["mask"].shape)

print("First row word ids:", toy_batch["word_ids"][0])
print("First row mask:", toy_batch["mask"][0])

word_ids shape: torch.Size([3, 8])
upos_ids shape: torch.Size([3, 8])
head_indices shape: torch.Size([3, 8])
mask shape: torch.Size([3, 8])
First row word ids: tensor([6301,    1,    2,    3,    4,    0,    0,    0])
First row mask: tensor([ True,  True,  True,  True,  True, False, False, False])


### Brief Discussion of Section 2B

Padding and masking are essential, otherwise sentences of different lengths cannot be grouped into mini-batches for efficient training.

The mask ensures that padding positions are excluded from both the loss computation and the evaluation metrics.

The toy batch check confirms that all tensors have the expected shapes and that padding appears in the correct positions.

## Section 2C: What the Parser Predicts

For each token, the parser chooses one head from all valid positions in the sentence, by assigning a score to every possible head-dependent pair.

In [18]:
# list gold arcs for one sentence
def list_gold_arcs(example):
    """Return the gold head-dependent arcs in one encoded sentence."""
    gold_arc_list = []

    token_count = len(example["token_text_list"])

    for dependent_index in range(1, token_count):
        head_index = example["head_index_list"][dependent_index]
        label_id = example["label_id_list"][dependent_index]
        label_name = id_to_label[label_id]

        gold_arc_list.append(
            {
                "head_index": head_index,
                "dependent_index": dependent_index,
                "head_token": example["token_text_list"][head_index],
                "dependent_token": example["token_text_list"][dependent_index],
                "label_name": label_name,
            }
        )

    return gold_arc_list

In [19]:
# print gold arcs for one Czech sentence
gold_arcs = list_gold_arcs(czech_train_examples[0])

for arc in gold_arcs:
    print(
        f"head={arc['head_index']} ({arc['head_token']}) -> "
        f"dep={arc['dependent_index']} ({arc['dependent_token']}), "
        f"label={arc['label_name']}"
    )

head=2 (rychlejší) -> dep=1 (Třikrát), label=obl
head=0 ([ROOT]) -> dep=2 (rychlejší), label=root
head=4 (slovo) -> dep=3 (než), label=mark
head=2 (rychlejší) -> dep=4 (slovo), label=advcl


In [20]:
def build_valid_arc_mask(example):
    """Build a matrix showing which head-dependent pairs are allowed."""
    token_count = len(example["token_text_list"])
    valid_arc_mask = torch.ones(token_count, token_count, dtype=torch.bool)

    valid_arc_mask[:, 0] = False
    for token_index in range(token_count):
        valid_arc_mask[token_index, token_index] = False

    return valid_arc_mask

In [21]:
valid_arc_mask = build_valid_arc_mask(czech_train_examples[0])

print("Valid arc mask shape:", valid_arc_mask.shape)
print(valid_arc_mask.int())

Valid arc mask shape: torch.Size([5, 5])
tensor([[0, 1, 1, 1, 1],
        [0, 0, 1, 1, 1],
        [0, 1, 0, 1, 1],
        [0, 1, 1, 0, 1],
        [0, 1, 1, 1, 0]], dtype=torch.int32)


### Brief Discussion of Section 2C

The gold arc listing confirms that the encoding correctly links each dependent token to its head with the right label.

The valid arc mask enforces two basic structural constraints, which are no self-loops and ROOT cannot be a dependent.

These constraints will later be applied to the parser's score matrix so that illegal arcs cannot receive high scores.

## Section 2D: Why Greedy Arc Choices Need MST

Choosing the best head for each token independently can produce cycles (e.g., token A picks B as head while B picks A).

Such cycles are not legal dependency trees, so we need MST decoding to find the highest-scoring tree that is globally consistent.

I implement MST decoding in Section 5B using the Edmonds/Chu-Liu algorithm from `networkx`.

## Section 3: A Simple Graph-Based Parser

This section builds the parser components, defines the losses, and checks that the training loop behaves as expected.


In [22]:
import torch.nn as nn

In [23]:
class SimpleTokenEncoder(nn.Module):
    """Create one vector per token by summing word, UPOS, and feature embeddings.
    This encoder is used for the initial sanity checks only;
    the configurable version in Section 6 handles the three representation modes.
    """
    def __init__(self, word_vocab_size, upos_vocab_size, feats_vocab_size, embedding_dim):
        super().__init__()
        self.word_embedding = nn.Embedding(word_vocab_size, embedding_dim)
        self.upos_embedding = nn.Embedding(upos_vocab_size, embedding_dim)
        self.feats_embedding = nn.Embedding(feats_vocab_size, embedding_dim)

    def forward(self, word_ids, upos_ids, feats_ids):
        word_vectors = self.word_embedding(word_ids)
        upos_vectors = self.upos_embedding(upos_ids)
        feats_vectors = self.feats_embedding(feats_ids)

        # sum the three embeddings to get one vevtor per token
        token_vectors = word_vectors + upos_vectors + feats_vectors
        return token_vectors

In [24]:
class SimpleArcScorer(nn.Module):
    """Score every possible head-dependent pair using a bilinear dot product.
    Separate linear projections create head and dependent representations,
    then their dot product gives the arc score matrix.
    """
    def __init__(self, representation_dim, hidden_dim):
        super().__init__()
        self.head_projection = nn.Linear(representation_dim, hidden_dim)
        self.dependent_projection = nn.Linear(representation_dim, hidden_dim)

    def forward(self, token_vectors):
        head_vectors = self.head_projection(token_vectors)
        dependent_vectors = self.dependent_projection(token_vectors)

        # score_matrix[h, d] = dot(head_vectors[h], dependent_vectors[d])
        score_matrix = torch.matmul(head_vectors, dependent_vectors.transpose(1, 2))
        return score_matrix

In [25]:
# check if encoder and arc scorer produce expected output shapes
embedding_dim = 64
arc_hidden_dim = 64

token_encoder = SimpleTokenEncoder(
    word_vocab_size=len(word_to_id),
    upos_vocab_size=len(upos_to_id),
    feats_vocab_size=len(feats_to_id),
    embedding_dim=embedding_dim,
)

arc_scorer = SimpleArcScorer(
    representation_dim=embedding_dim,
    hidden_dim=arc_hidden_dim,
)

toy_batch = pad_example_list(czech_train_examples[:2])

token_vectors = token_encoder(
    toy_batch["word_ids"],
    toy_batch["upos_ids"],
    toy_batch["feats_ids"],
)

arc_score_matrix = arc_scorer(token_vectors)

print("word_ids shape:", toy_batch["word_ids"].shape)
print("token_vectors shape:", token_vectors.shape)
print("arc_score_matrix shape:", arc_score_matrix.shape)

word_ids shape: torch.Size([2, 8])
token_vectors shape: torch.Size([2, 8, 64])
arc_score_matrix shape: torch.Size([2, 8, 8])


### Brief Discussion of Section 3A

The model now produces one score for every possible head-dependent pair in a sentence, which is the core output of a graph-based dependency parser.

At this stage the scores are still random because the model has not been trained yet.

The next step is to mask illegal arcs and define a loss that teaches the model to choose the correct head for each token.

In [26]:
import torch.nn.functional as F

def build_batch_arc_mask(batch_mask):
    """Build a legal arc mask for a padded batch."""
    sequence_length = batch_mask.size(1)

    valid_head_mask = batch_mask.unsqueeze(2)
    valid_dependent_mask = batch_mask.unsqueeze(1)
    valid_arc_mask = valid_head_mask & valid_dependent_mask

    valid_arc_mask[:, :, 0] = False # ROOT cannot be a dependent

    diagonal_mask = torch.eye(sequence_length, dtype=torch.bool, device=batch_mask.device)
    valid_arc_mask = valid_arc_mask & ~diagonal_mask.unsqueeze(0)

    return valid_arc_mask

In [27]:
batch_valid_arc_mask = build_batch_arc_mask(toy_batch["mask"])

print("batch_valid_arc_mask shape:", batch_valid_arc_mask.shape)
print("First sentence valid arc mask:")
print(batch_valid_arc_mask[0].int())

batch_valid_arc_mask shape: torch.Size([2, 8, 8])
First sentence valid arc mask:
tensor([[0, 1, 1, 1, 1, 0, 0, 0],
        [0, 0, 1, 1, 1, 0, 0, 0],
        [0, 1, 0, 1, 1, 0, 0, 0],
        [0, 1, 1, 0, 1, 0, 0, 0],
        [0, 1, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.int32)


In [28]:
def mask_invalid_arcs(arc_score_matrix, valid_arc_mask):
    """Set illegal arc scores to -1e9 so they get near-zero probability after softmax."""
    return arc_score_matrix.masked_fill(~valid_arc_mask, -1e9)

def compute_arc_loss(arc_score_matrix, gold_head_indices, valid_arc_mask):
    """Compute cross-entropy loss over head predictions for one batch."""
    masked_arc_score_matrix = mask_invalid_arcs(arc_score_matrix, valid_arc_mask)

    arc_loss = F.cross_entropy(
        masked_arc_score_matrix,
        gold_head_indices,
        ignore_index=ignore_index,
    )

    return arc_loss, masked_arc_score_matrix

In [29]:
# verify arc loss computes without errors
arc_loss, masked_arc_score_matrix = compute_arc_loss(
    arc_score_matrix,
    toy_batch["head_indices"],
    batch_valid_arc_mask,
)

print("Gold heads for first sentence:", toy_batch["head_indices"][0])
print("Masked arc score matrix shape:", masked_arc_score_matrix.shape)
print("Arc loss:", arc_loss.item())

Gold heads for first sentence: tensor([-1,  2,  0,  4,  2, -1, -1, -1])
Masked arc score matrix shape: torch.Size([2, 8, 8])
Arc loss: 10.040029525756836


### Brief Discussion of Section 3B

The arc loss teaches the parser to choose the correct head for each token.
However, dependency parsing also needs a label for each arc, such as `nsubj` or `obj`.

In this step, I train a simple label classifier using the gold head of each token.

This separates the two learning problems and keeps the first parser version easy to understand.

In [30]:
def gather_gold_head_vectors(token_vectors, gold_head_indices):
    """Collect the vector of the gold head for each token."""
    batch_size, sequence_length, _ = token_vectors.shape

    # padding positions use -1, so I clamp them to 0 before indexing
    safe_head_indices = gold_head_indices.clamp(min=0)

    batch_index_matrix = torch.arange(batch_size, device=token_vectors.device)
    batch_index_matrix = batch_index_matrix.unsqueeze(1).expand(batch_size, sequence_length)

    head_vectors = token_vectors[batch_index_matrix, safe_head_indices]
    return head_vectors

In [31]:
class SimpleLabelScorer(nn.Module):
    """Predict the dependency label for a head-dependent pair.
    Concatenates head and dependent vectors, then applies a linear classifier.
    """
    def __init__(self, representation_dim, label_count):
        super().__init__()
        self.label_classifier = nn.Linear(representation_dim * 2, label_count)

    def forward(self, head_vectors, dependent_vectors):
        pair_vectors = torch.cat([head_vectors, dependent_vectors], dim=-1)
        label_logits = self.label_classifier(pair_vectors)
        return label_logits

In [32]:
# check if label scorer produces logits over all dependency labels
label_scorer = SimpleLabelScorer(
    representation_dim=embedding_dim,
    label_count=len(label_to_id),
)

gold_head_vectors = gather_gold_head_vectors(
    token_vectors,
    toy_batch["head_indices"],
)

label_logits = label_scorer(
    gold_head_vectors,
    token_vectors,
)

print("gold_head_vectors shape:", gold_head_vectors.shape)
print("label_logits shape:", label_logits.shape)

gold_head_vectors shape: torch.Size([2, 8, 64])
label_logits shape: torch.Size([2, 8, 61])


In [33]:
def compute_label_loss(label_logits, gold_label_ids):
    """Compute cross-entropy loss over dependency label predictions.
    Flattens the batch and sequence dimensions so that F.cross_entropy can handle it.
    """
    flat_label_logits = label_logits.reshape(-1, label_logits.size(-1))
    flat_gold_label_ids = gold_label_ids.reshape(-1)

    label_loss = F.cross_entropy(
        flat_label_logits,
        flat_gold_label_ids,
        ignore_index=ignore_index,
    )

    return label_loss

In [34]:
# test label loss and total loss, total loss is the sum of arc loss and label loss
label_loss = compute_label_loss(
    label_logits,
    toy_batch["label_ids"],
)

total_loss = arc_loss + label_loss

print("Gold labels for first sentence:", toy_batch["label_ids"][0])
print("Label loss:", label_loss.item())
print("Total loss:", total_loss.item())

Gold labels for first sentence: tensor([-1,  2,  3,  4,  5, -1, -1, -1])
Label loss: 4.358348846435547
Total loss: 14.398378372192383


### Brief Discussion of Section 3C

The model can now score dependency arcs and dependency labels.

The next check is whether these losses can actually decrease during optimization.

To keep this step simple, I train on a very small batch and verify that the model can overfit it.

If this works, the parser pipeline is connected correctly.

In [35]:
def compute_total_loss(batch, token_encoder, arc_scorer, label_scorer):
    """Run one forward pass and return arc loss, label loss, and total loss."""
    token_vectors = token_encoder(
        batch["word_ids"],
        batch["upos_ids"],
        batch["feats_ids"],
    )

    arc_score_matrix = arc_scorer(token_vectors)
    valid_arc_mask = build_batch_arc_mask(batch["mask"])

    arc_loss, _ = compute_arc_loss(
        arc_score_matrix,
        batch["head_indices"],
        valid_arc_mask,
    )

    gold_head_vectors = gather_gold_head_vectors(
        token_vectors,
        batch["head_indices"],
    )

    label_logits = label_scorer(
        gold_head_vectors,
        token_vectors,
    )

    label_loss = compute_label_loss(
        label_logits,
        batch["label_ids"],
    )

    total_loss = arc_loss + label_loss
    return arc_loss, label_loss, total_loss

In [36]:
# reinitialize models and optimizer for the overfitting test
embedding_dim = 64
arc_hidden_dim = 64

token_encoder = SimpleTokenEncoder(
    word_vocab_size=len(word_to_id),
    upos_vocab_size=len(upos_to_id),
    feats_vocab_size=len(feats_to_id),
    embedding_dim=embedding_dim,
)

arc_scorer = SimpleArcScorer(
    representation_dim=embedding_dim,
    hidden_dim=arc_hidden_dim,
)

label_scorer = SimpleLabelScorer(
    representation_dim=embedding_dim,
    label_count=len(label_to_id),
)

optimizer = torch.optim.Adam(
    list(token_encoder.parameters()) +
    list(arc_scorer.parameters()) +
    list(label_scorer.parameters()),
    lr=0.005,
)

tiny_batch = pad_example_list(czech_train_examples[:8])

In [37]:
# the overfitting training loop
loss_history = []

for step_index in range(60):
    optimizer.zero_grad()

    arc_loss, label_loss, total_loss = compute_total_loss(
        tiny_batch,
        token_encoder,
        arc_scorer,
        label_scorer,
    )

    total_loss.backward()
    optimizer.step()

    loss_history.append(total_loss.item())

    if step_index % 10 == 0 or step_index == 59:
        print(
            f"step={step_index:02d} "
            f"arc_loss={arc_loss.item():.4f} "
            f"label_loss={label_loss.item():.4f} "
            f"total_loss={total_loss.item():.4f}"
        )

step=00 arc_loss=13.7600 label_loss=4.3191 total_loss=18.0791
step=10 arc_loss=0.1712 label_loss=0.3983 total_loss=0.5695
step=20 arc_loss=0.1239 label_loss=0.0592 total_loss=0.1831
step=30 arc_loss=0.1227 label_loss=0.0206 total_loss=0.1433
step=40 arc_loss=0.1165 label_loss=0.0118 total_loss=0.1283
step=50 arc_loss=0.1119 label_loss=0.0087 total_loss=0.1206
step=59 arc_loss=0.1116 label_loss=0.0074 total_loss=0.1189


### Brief Discussion of Section 3D

The tiny-batch overfitting test passed, so the training pipeline is working correctly.

The loss dropped sharply on a very small batch, which confirms that forward pass, loss computation, and optimization are connected correctly.

This stage is only a sanity check and does not measure real generalization.

The next step is to train on a larger subset and evaluate on development data.

In [38]:
def create_batches(example_list, batch_size, shuffle=True):
    """Group encoded sentences into padded mini-batches."""
    example_indices = list(range(len(example_list)))

    if shuffle:
        random.shuffle(example_indices)

    batch_list = []

    for start_index in range(0, len(example_indices), batch_size):
        batch_indices = example_indices[start_index:start_index + batch_size]
        batch_examples = [example_list[index] for index in batch_indices]
        batch = pad_example_list(batch_examples)
        batch_list.append(batch)

    return batch_list

In [39]:
def train_one_epoch(example_list, batch_size, token_encoder, arc_scorer, label_scorer, optimizer):
    """Train for one epoch and return average losses."""
    token_encoder.train()
    arc_scorer.train()
    label_scorer.train()

    batch_list = create_batches(example_list, batch_size=batch_size, shuffle=True)

    total_arc_loss = 0.0
    total_label_loss = 0.0
    total_loss_value = 0.0

    for batch in batch_list:
        optimizer.zero_grad()

        arc_loss, label_loss, total_loss = compute_total_loss(
            batch,
            token_encoder,
            arc_scorer,
            label_scorer,
        )

        total_loss.backward()
        optimizer.step()

        total_arc_loss += arc_loss.item()
        total_label_loss += label_loss.item()
        total_loss_value += total_loss.item()

    batch_count = len(batch_list)

    return {
        "average_arc_loss": total_arc_loss / batch_count,
        "average_label_loss": total_label_loss / batch_count,
        "average_total_loss": total_loss_value / batch_count,
    }

### Brief Discussion of Section 3E

The parser now has all core pieces needed for a full experiment: batching, scoring, losses, and a standard PyTorch update loop.

The tiny-batch overfitting check is only a sanity test, but it shows that forward pass, backward pass, and optimization are connected correctly.

From this point on, I switch to real train/dev/test splits and formal parsing metrics.

The next section builds a simple greedy baseline before moving to MST decoding.

## Section 4: Greedy Baseline on Czech Dev Data

I first train a simple baseline on Czech and evaluate it with greedy head selection so I have a working reference point before MST decoding.


In [40]:
def gather_head_vectors(token_vectors, head_indices):
    """Collect the vector of the chosen (predicted) head for each token.
    Same logic as gather_gold_head_vectors, but used at inference time.
    """
    batch_size, sequence_length, _ = token_vectors.shape

    safe_head_indices = head_indices.clamp(min=0)

    batch_index_matrix = torch.arange(batch_size, device=token_vectors.device)
    batch_index_matrix = batch_index_matrix.unsqueeze(1).expand(batch_size, sequence_length)

    head_vectors = token_vectors[batch_index_matrix, safe_head_indices]
    return head_vectors

In [41]:
def predict_batch_greedily(batch, token_encoder, arc_scorer, label_scorer):
    """Predict heads and labels for one batch using greedy (argmax) decoding.
    Each token independently picks its highest-scoring head.
    This does not guarantee a valid tree — it serves as a baseline for MST decoding.
    """
    token_vectors = token_encoder(
        batch["word_ids"],
        batch["upos_ids"],
        batch["feats_ids"],
    )

    arc_score_matrix = arc_scorer(token_vectors)
    valid_arc_mask = build_batch_arc_mask(batch["mask"])
    masked_arc_score_matrix = mask_invalid_arcs(arc_score_matrix, valid_arc_mask)

    predicted_head_indices = torch.argmax(masked_arc_score_matrix, dim=1)
    predicted_head_indices[:, 0] = ignore_index
    predicted_head_indices[~batch["mask"]] = ignore_index

    predicted_head_vectors = gather_head_vectors(token_vectors, predicted_head_indices)
    label_logits = label_scorer(predicted_head_vectors, token_vectors)

    predicted_label_ids = torch.argmax(label_logits, dim=-1)
    predicted_label_ids[:, 0] = ignore_index
    predicted_label_ids[~batch["mask"]] = ignore_index

    return {
        "predicted_head_indices": predicted_head_indices,
        "predicted_label_ids": predicted_label_ids,
    }

In [42]:
def evaluate_greedy_parser(example_list, batch_size, token_encoder, arc_scorer, label_scorer):
    """Evaluate greedy predictions and return head, label, and joint accuracy.
    Joint accuracy here means both head and label are correct for a token.
    """
    token_encoder.eval()
    arc_scorer.eval()
    label_scorer.eval()

    batch_list = create_batches(example_list, batch_size=batch_size, shuffle=False)

    total_token_count = 0
    total_head_correct = 0
    total_label_correct = 0
    total_joint_correct = 0

    with torch.no_grad():
        for batch in batch_list:
            prediction_dict = predict_batch_greedily(
                batch,
                token_encoder,
                arc_scorer,
                label_scorer,
            )

            valid_token_mask = batch["head_indices"] != ignore_index

            head_correct_mask = (
                prediction_dict["predicted_head_indices"] == batch["head_indices"]
            ) & valid_token_mask

            label_correct_mask = (
                prediction_dict["predicted_label_ids"] == batch["label_ids"]
            ) & valid_token_mask

            joint_correct_mask = head_correct_mask & label_correct_mask

            total_token_count += valid_token_mask.sum().item()
            total_head_correct += head_correct_mask.sum().item()
            total_label_correct += label_correct_mask.sum().item()
            total_joint_correct += joint_correct_mask.sum().item()

    return {
        "head_accuracy": total_head_correct / total_token_count,
        "label_accuracy": total_label_correct / total_token_count,
        "joint_accuracy": total_joint_correct / total_token_count,
        "token_count": total_token_count,
    }

In [43]:
# train a greedy baseline on Czech to verify the full pipeline before MST decoding
train_example_list = czech_train_examples[:2000]
dev_example_list = czech_dev_examples[:500]

batch_size = 32
epoch_count = 12

embedding_dim = 64
arc_hidden_dim = 64

token_encoder = SimpleTokenEncoder(
    word_vocab_size=len(word_to_id),
    upos_vocab_size=len(upos_to_id),
    feats_vocab_size=len(feats_to_id),
    embedding_dim=embedding_dim,
)

arc_scorer = SimpleArcScorer(
    representation_dim=embedding_dim,
    hidden_dim=arc_hidden_dim,
)

label_scorer = SimpleLabelScorer(
    representation_dim=embedding_dim,
    label_count=len(label_to_id),
)

optimizer = torch.optim.Adam(
    list(token_encoder.parameters()) +
    list(arc_scorer.parameters()) +
    list(label_scorer.parameters()),
    lr=0.003,
)

for epoch_index in range(epoch_count):
    train_loss_dict = train_one_epoch(
        train_example_list,
        batch_size,
        token_encoder,
        arc_scorer,
        label_scorer,
        optimizer,
    )

    dev_metrics = evaluate_greedy_parser(
        dev_example_list,
        batch_size,
        token_encoder,
        arc_scorer,
        label_scorer,
    )

    print(
        f"epoch={epoch_index + 1} "
        f"train_total_loss={train_loss_dict['average_total_loss']:.4f} "
        f"dev_head={dev_metrics['head_accuracy']:.4f} "
        f"dev_joint={dev_metrics['joint_accuracy']:.4f}"
    )

epoch=1 train_total_loss=5.5149 dev_head=0.3449 dev_joint=0.2996
epoch=2 train_total_loss=2.7896 dev_head=0.3784 dev_joint=0.3373
epoch=3 train_total_loss=2.5364 dev_head=0.3935 dev_joint=0.3553
epoch=4 train_total_loss=2.3937 dev_head=0.4150 dev_joint=0.3729
epoch=5 train_total_loss=2.2694 dev_head=0.4171 dev_joint=0.3797
epoch=6 train_total_loss=2.1698 dev_head=0.4321 dev_joint=0.3938
epoch=7 train_total_loss=2.0605 dev_head=0.4414 dev_joint=0.4064
epoch=8 train_total_loss=1.9618 dev_head=0.4447 dev_joint=0.4093
epoch=9 train_total_loss=1.8888 dev_head=0.4507 dev_joint=0.4135
epoch=10 train_total_loss=1.8088 dev_head=0.4574 dev_joint=0.4206
epoch=11 train_total_loss=1.7355 dev_head=0.4557 dev_joint=0.4185
epoch=12 train_total_loss=1.6649 dev_head=0.4631 dev_joint=0.4260


### Brief Discussion of Section 4 (Greedy Evaluation)

The greedy baseline is useful as a first end-to-end check because it combines training, decoding, and evaluation in one simple setup.

Its main weakness is structural. It predicts the best head for each token independently, so it does not guarantee a legal dependency tree.

That means its scores should be treated as a baseline, not as the final parser result.

The next step is to switch to formal UAS/LAS and then replace greedy decoding with MST decoding.

## Section 5A: Formal UAS/LAS Under Greedy Decoding

I now use standard parsing metrics before introducing MST decoding.


In [44]:
def evaluate_greedy_uas_las(
    example_list,
    batch_size,
    token_encoder,
    arc_scorer,
    label_scorer,
    ignore_punct=False,
):
    """Evaluate greedy decoding with standard UAS and LAS metrics.
    UAS = fraction of tokens with correct head.
    LAS = fraction of tokens with both correct head and correct label.
    """
    token_encoder.eval()
    arc_scorer.eval()
    label_scorer.eval()

    punct_upos_id = upos_to_id.get("PUNCT", None)

    total_eval_token_count = 0
    total_head_correct_count = 0
    total_las_correct_count = 0

    batch_list = create_batches(example_list, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch in batch_list:
            prediction_dict = predict_batch_greedily(
                batch,
                token_encoder,
                arc_scorer,
                label_scorer,
            )

            eval_mask = batch["head_indices"] != ignore_index
            eval_mask[:, 0] = False  # Exclude ROOT as dependent.

            if ignore_punct and punct_upos_id is not None:
                eval_mask = eval_mask & (batch["upos_ids"] != punct_upos_id)

            head_correct_mask = (
                prediction_dict["predicted_head_indices"] == batch["head_indices"]
            ) & eval_mask

            las_correct_mask = (
                prediction_dict["predicted_label_ids"] == batch["label_ids"]
            ) & head_correct_mask

            total_eval_token_count += eval_mask.sum().item()
            total_head_correct_count += head_correct_mask.sum().item()
            total_las_correct_count += las_correct_mask.sum().item()

    uas = total_head_correct_count / total_eval_token_count
    las = total_las_correct_count / total_eval_token_count

    return {
        "UAS": uas,
        "LAS": las,
        "token_count": total_eval_token_count,
    }

In [45]:
# report greedy UAS/LAS both with and without punctuation tokens
greedy_dev_with_punct = evaluate_greedy_uas_las(
    dev_example_list,
    batch_size=32,
    token_encoder=token_encoder,
    arc_scorer=arc_scorer,
    label_scorer=label_scorer,
    ignore_punct=False,
)

greedy_dev_without_punct = evaluate_greedy_uas_las(
    dev_example_list,
    batch_size=32,
    token_encoder=token_encoder,
    arc_scorer=arc_scorer,
    label_scorer=label_scorer,
    ignore_punct=True,
)

print(f"Greedy Dev UAS (with punct): {greedy_dev_with_punct['UAS']:.4f}")
print(f"Greedy Dev LAS (with punct): {greedy_dev_with_punct['LAS']:.4f}")
print(f"Greedy Dev UAS (without punct): {greedy_dev_without_punct['UAS']:.4f}")
print(f"Greedy Dev LAS (without punct): {greedy_dev_without_punct['LAS']:.4f}")

Greedy Dev UAS (with punct): 0.4631
Greedy Dev LAS (with punct): 0.4260
Greedy Dev UAS (without punct): 0.4692
Greedy Dev LAS (without punct): 0.4252


### Brief Discussion of Section 5A (Greedy UAS/LAS)

Greedy UAS/LAS gives a cleaner baseline than raw head or joint accuracy because these are the standard dependency parsing metrics.

The only thing that changes in Section 5B is the decoding method.

If MST decoding helps, the gain can be interpreted as the value of enforcing a legal tree structure.

## Section 5B: MST Decoding + UAS/LAS
Replace greedy head selection with MST decoding, then compare UAS/LAS.

In [46]:
# install networkx if needed, used for the Edmonds/Chu-Liu MST algorithm.
try:
    import networkx as nx
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "networkx"])
    import networkx as nx

from networkx.algorithms.tree.branchings import maximum_spanning_arborescence

In [47]:
def decode_mst_heads_for_sentence(arc_score_matrix, token_mask):
    """Find the highest-scoring dependency tree for one sentence using MST.
    A super-root node is added to guarantee that the real ROOT (index 0) is
    always selected, following the standard Edmonds/Chu-Liu approach.
    """
    sentence_length = int(token_mask.sum().item())
    sentence_score_matrix = arc_score_matrix[:sentence_length, :sentence_length]

    graph = nx.DiGraph()
    super_root_index = sentence_length

    graph.add_nodes_from(range(sentence_length + 1))
    # force super-root to pick real ROOT with a very high weight
    graph.add_edge(super_root_index, 0, weight=1e9)

    for dependent_index in range(1, sentence_length):
        # discourage super-root from directly attaching real tokens
        graph.add_edge(super_root_index, dependent_index, weight=-1e9)
        for head_index in range(sentence_length):
            if head_index == dependent_index:
                continue
            arc_score = sentence_score_matrix[head_index, dependent_index].item()
            graph.add_edge(head_index, dependent_index, weight=arc_score)

    mst_tree = maximum_spanning_arborescence(graph, attr="weight", preserve_attrs=True)

    predicted_head_list = [ignore_index] * sentence_length

    for head_index, dependent_index in mst_tree.edges():
        if dependent_index == 0 or dependent_index == super_root_index:
            continue
        predicted_head_list[dependent_index] = 0 if head_index == super_root_index else head_index

    return predicted_head_list

In [48]:
def predict_batch_with_mst(batch, token_encoder, arc_scorer, label_scorer):
    """Predict heads with MST decoding and labels from predicted heads."""
    token_vectors = token_encoder(
        batch["word_ids"],
        batch["upos_ids"],
        batch["feats_ids"],
    )

    arc_score_matrix = arc_scorer(token_vectors)

    batch_size, sequence_length, _ = arc_score_matrix.shape
    predicted_head_indices = torch.full(
        (batch_size, sequence_length),
        fill_value=ignore_index,
        dtype=torch.long,
        device=arc_score_matrix.device,
    )

    # MST decoding runs per sentence because networkx works on single graphs
    for batch_index in range(batch_size):
        sentence_head_list = decode_mst_heads_for_sentence(
            arc_score_matrix[batch_index],
            batch["mask"][batch_index],
        )
        sentence_length = len(sentence_head_list)
        predicted_head_indices[batch_index, :sentence_length] = torch.tensor(
            sentence_head_list,
            dtype=torch.long,
            device=arc_score_matrix.device,
        )

    # predict labels using the MST-decoded heads
    predicted_head_vectors = gather_head_vectors(token_vectors, predicted_head_indices)
    label_logits = label_scorer(predicted_head_vectors, token_vectors)
    predicted_label_ids = torch.argmax(label_logits, dim=-1)

    predicted_label_ids[:, 0] = ignore_index
    predicted_label_ids[~batch["mask"]] = ignore_index

    return {
        "predicted_head_indices": predicted_head_indices,
        "predicted_label_ids": predicted_label_ids,
    }

In [49]:
def evaluate_mst_uas_las(
    example_list,
    batch_size,
    token_encoder,
    arc_scorer,
    label_scorer,
    ignore_punct=False,
):
    """Evaluate MST decoding with standard UAS/LAS.
    Same logic as evaluate_greedy_uas_las but uses MST instead of argmax.
    """
    token_encoder.eval()
    arc_scorer.eval()
    label_scorer.eval()

    punct_upos_id = upos_to_id.get("PUNCT", None)

    total_eval_token_count = 0
    total_head_correct_count = 0
    total_las_correct_count = 0

    batch_list = create_batches(example_list, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch in batch_list:
            prediction_dict = predict_batch_with_mst(
                batch,
                token_encoder,
                arc_scorer,
                label_scorer,
            )

            eval_mask = batch["head_indices"] != ignore_index
            eval_mask[:, 0] = False

            if ignore_punct and punct_upos_id is not None:
                eval_mask = eval_mask & (batch["upos_ids"] != punct_upos_id)

            head_correct_mask = (
                prediction_dict["predicted_head_indices"] == batch["head_indices"]
            ) & eval_mask

            las_correct_mask = (
                prediction_dict["predicted_label_ids"] == batch["label_ids"]
            ) & head_correct_mask

            total_eval_token_count += eval_mask.sum().item()
            total_head_correct_count += head_correct_mask.sum().item()
            total_las_correct_count += las_correct_mask.sum().item()

    return {
        "UAS": total_head_correct_count / total_eval_token_count,
        "LAS": total_las_correct_count / total_eval_token_count,
        "token_count": total_eval_token_count,
    }

In [50]:
# compute MST UAS/LAS and compare with the greedy baseline on Czech dev
mst_dev_with_punct = evaluate_mst_uas_las(
    dev_example_list,
    batch_size=32,
    token_encoder=token_encoder,
    arc_scorer=arc_scorer,
    label_scorer=label_scorer,
    ignore_punct=False,
)

mst_dev_without_punct = evaluate_mst_uas_las(
    dev_example_list,
    batch_size=32,
    token_encoder=token_encoder,
    arc_scorer=arc_scorer,
    label_scorer=label_scorer,
    ignore_punct=True,
)

print(f"MST Dev UAS (with punct): {mst_dev_with_punct['UAS']:.4f}")
print(f"MST Dev LAS (with punct): {mst_dev_with_punct['LAS']:.4f}")
print(f"MST Dev UAS (without punct): {mst_dev_without_punct['UAS']:.4f}")
print(f"MST Dev LAS (without punct): {mst_dev_without_punct['LAS']:.4f}")

print(f"Delta UAS (with punct, MST-Greedy): {mst_dev_with_punct['UAS'] - greedy_dev_with_punct['UAS']:.4f}")
print(f"Delta LAS (with punct, MST-Greedy): {mst_dev_with_punct['LAS'] - greedy_dev_with_punct['LAS']:.4f}")

MST Dev UAS (with punct): 0.4698
MST Dev LAS (with punct): 0.4325
MST Dev UAS (without punct): 0.4768
MST Dev LAS (without punct): 0.4327
Delta UAS (with punct, MST-Greedy): 0.0067
Delta LAS (with punct, MST-Greedy): 0.0066


### Brief Discussion of Section 5B (MST Decoding)

MST decoding is the main decoding method for the final project because it returns a highest-scoring legal tree rather than independent local choices.

After decoding step is in place, it is ready for comparing lexicalized and morphosyntactic input settings across Czech and English.

## Section 6: Held-Out Test Comparison Across Representations

For each language, I train three versions of the same parser:
1. lexicalized
2. morphosyntax-only
3. mixed

I first define a configurable token encoder that switches between the three input settings, then choose the best epoch on the development set and report final UAS/LAS on held-out test data.

In [51]:
class ConfigurableTokenEncoder(nn.Module):
    """Create token vectors for lexicalized, morphosyntax-only, or mixed input."""
    def __init__(
        self,
        word_vocab_size,
        upos_vocab_size,
        feats_vocab_size,
        embedding_dim,
        representation_mode,
    ):
        super().__init__()
        self.word_embedding = nn.Embedding(word_vocab_size, embedding_dim)
        self.upos_embedding = nn.Embedding(upos_vocab_size, embedding_dim)
        self.feats_embedding = nn.Embedding(feats_vocab_size, embedding_dim)
        self.representation_mode = representation_mode

    def forward(self, word_ids, upos_ids, feats_ids):
        word_vectors = self.word_embedding(word_ids)
        upos_vectors = self.upos_embedding(upos_ids)
        feats_vectors = self.feats_embedding(feats_ids)

        token_vectors = torch.zeros_like(word_vectors)

        if self.representation_mode in {"lexicalized", "mixed"}:
            token_vectors = token_vectors + word_vectors

        if self.representation_mode in {"morphosyntax_only", "mixed"}:
            token_vectors = token_vectors + upos_vectors + feats_vectors

        return token_vectors

In [52]:
def build_parser_modules(representation_mode, embedding_dim=64, arc_hidden_dim=64):
    """Build one parser configuration for a specific input representation."""
    token_encoder = ConfigurableTokenEncoder(
        word_vocab_size=len(word_to_id),
        upos_vocab_size=len(upos_to_id),
        feats_vocab_size=len(feats_to_id),
        embedding_dim=embedding_dim,
        representation_mode=representation_mode,
    )

    arc_scorer = SimpleArcScorer(
        representation_dim=embedding_dim,
        hidden_dim=arc_hidden_dim,
    )

    label_scorer = SimpleLabelScorer(
        representation_dim=embedding_dim,
        label_count=len(label_to_id),
    )

    return token_encoder, arc_scorer, label_scorer

In [53]:
# test if all three representation modes produce same output shape
for representation_mode in ["lexicalized", "morphosyntax_only", "mixed"]:
    test_token_encoder, _, _ = build_parser_modules(representation_mode)

    test_token_vectors = test_token_encoder(
        toy_batch["word_ids"],
        toy_batch["upos_ids"],
        toy_batch["feats_ids"],
    )

    print(representation_mode, test_token_vectors.shape)

lexicalized torch.Size([2, 8, 64])
morphosyntax_only torch.Size([2, 8, 64])
mixed torch.Size([2, 8, 64])


In [54]:
def train_and_select_representation(
    representation_mode,
    train_example_list,
    dev_example_list,
    test_example_list,
    batch_size=32,
    epoch_count=8,
    learning_rate=0.003,
):
    """Train one parser setting, select the best epoch by dev LAS, then evaluate on test.
    Returns the best dev result, test UAS/LAS, and the trained model modules.
    """
    token_encoder, arc_scorer, label_scorer = build_parser_modules(
        representation_mode=representation_mode,
        embedding_dim=64,
        arc_hidden_dim=64,
    )

    optimizer = torch.optim.Adam(
        list(token_encoder.parameters()) +
        list(arc_scorer.parameters()) +
        list(label_scorer.parameters()),
        lr=learning_rate,
    )

    best_dev_result = None
    best_state = None
    epoch_history = []

    for epoch_index in range(epoch_count):
        train_loss_dict = train_one_epoch(
            train_example_list,
            batch_size,
            token_encoder,
            arc_scorer,
            label_scorer,
            optimizer,
        )

        dev_metrics = evaluate_mst_uas_las(
            dev_example_list,
            batch_size=batch_size,
            token_encoder=token_encoder,
            arc_scorer=arc_scorer,
            label_scorer=label_scorer,
            ignore_punct=True,
        )

        current_result = {
            "epoch": epoch_index + 1,
            "train_total_loss": train_loss_dict["average_total_loss"],
            "dev_uas": dev_metrics["UAS"],
            "dev_las": dev_metrics["LAS"],
        }
        epoch_history.append(current_result)

        # keep the checkpoint with the highest dev LAS
        if best_dev_result is None or current_result["dev_las"] > best_dev_result["dev_las"]:
            best_dev_result = current_result
            best_state = {
                "token_encoder": copy.deepcopy(token_encoder.state_dict()),
                "arc_scorer": copy.deepcopy(arc_scorer.state_dict()),
                "label_scorer": copy.deepcopy(label_scorer.state_dict()),
            }

        print(
            f"{representation_mode} "
            f"epoch={epoch_index + 1} "
            f"train_total_loss={current_result['train_total_loss']:.4f} "
            f"dev_uas={current_result['dev_uas']:.4f} "
            f"dev_las={current_result['dev_las']:.4f}"
        )

    # restore the best checkpoint and evaluate on held-out test data
    token_encoder.load_state_dict(best_state["token_encoder"])
    arc_scorer.load_state_dict(best_state["arc_scorer"])
    label_scorer.load_state_dict(best_state["label_scorer"])

    test_metrics = evaluate_mst_uas_las(
        test_example_list,
        batch_size=batch_size,
        token_encoder=token_encoder,
        arc_scorer=arc_scorer,
        label_scorer=label_scorer,
        ignore_punct=True,
    )

    return {
        "representation_mode": representation_mode,
        "best_dev_result": best_dev_result,
        "test_metrics": test_metrics,
        "epoch_history": epoch_history,
        "token_encoder": token_encoder,
        "arc_scorer": arc_scorer,
        "label_scorer": label_scorer,
    }

In [55]:
# train all three representation settings on Czech
czech_train_for_comparison = czech_train_examples[:2000]
czech_dev_for_comparison = czech_dev_examples[:500]
czech_test_for_comparison = czech_test_examples[:500]

czech_results = []

for representation_mode in ["lexicalized", "morphosyntax_only", "mixed"]:
    experiment_result = train_and_select_representation(
        representation_mode=representation_mode,
        train_example_list=czech_train_for_comparison,
        dev_example_list=czech_dev_for_comparison,
        test_example_list=czech_test_for_comparison,
        batch_size=32,
        epoch_count=8,
        learning_rate=0.003,
    )
    czech_results.append(experiment_result)

lexicalized epoch=1 train_total_loss=6.1317 dev_uas=0.1727 dev_las=0.0646
lexicalized epoch=2 train_total_loss=4.7042 dev_uas=0.1807 dev_las=0.0791
lexicalized epoch=3 train_total_loss=4.2982 dev_uas=0.1588 dev_las=0.1039
lexicalized epoch=4 train_total_loss=3.9955 dev_uas=0.1749 dev_las=0.1224
lexicalized epoch=5 train_total_loss=3.7438 dev_uas=0.2257 dev_las=0.1260
lexicalized epoch=6 train_total_loss=3.5035 dev_uas=0.2328 dev_las=0.1311
lexicalized epoch=7 train_total_loss=3.3085 dev_uas=0.2038 dev_las=0.1488
lexicalized epoch=8 train_total_loss=3.1411 dev_uas=0.2487 dev_las=0.1508
morphosyntax_only epoch=1 train_total_loss=4.2849 dev_uas=0.4173 dev_las=0.3653
morphosyntax_only epoch=2 train_total_loss=2.5598 dev_uas=0.4294 dev_las=0.3782
morphosyntax_only epoch=3 train_total_loss=2.3609 dev_uas=0.4428 dev_las=0.3954
morphosyntax_only epoch=4 train_total_loss=2.2594 dev_uas=0.4538 dev_las=0.4026
morphosyntax_only epoch=5 train_total_loss=2.1807 dev_uas=0.4669 dev_las=0.4198
morphosy

In [56]:
for experiment_result in czech_results:
    best_dev_result = experiment_result["best_dev_result"]
    test_metrics = experiment_result["test_metrics"]
    print(
        f"{experiment_result['representation_mode']}: "
        f"best_dev_epoch={best_dev_result['epoch']} "
        f"best_dev_uas={best_dev_result['dev_uas']:.4f} "
        f"best_dev_las={best_dev_result['dev_las']:.4f} "
        f"test_uas={test_metrics['UAS']:.4f} "
        f"test_las={test_metrics['LAS']:.4f}"
    )

lexicalized: best_dev_epoch=8 best_dev_uas=0.2487 best_dev_las=0.1508 test_uas=0.2189 test_las=0.1318
morphosyntax_only: best_dev_epoch=8 best_dev_uas=0.4754 best_dev_las=0.4327 test_uas=0.4382 test_las=0.4017
mixed: best_dev_epoch=8 best_dev_uas=0.4480 best_dev_las=0.4022 test_uas=0.4212 test_las=0.3749


### Brief Discussion of Section 6A (Czech Test Comparison)

On held-out Czech test data, the morphosyntax-only setting performed best with LAS 0.4017, while mixed reached 0.3749 and lexicalized stayed much lower at 0.1318.

The gap between morphosyntax-only and lexicalized is large, about 0.27 LAS, so morphosyntactic information alone is clearly very useful for Czech in this parser.

The mixed setting is also much better than lexicalized, but it still does not beat morphosyntax-only.

This means the Czech results strongly support the project idea that overt morphology carries important syntactic cues.

In [57]:
# train all three representation settings on English
english_train_for_comparison = english_train_examples[:2000]
english_dev_for_comparison = english_dev_examples[:500]
english_test_for_comparison = english_test_examples[:500]

english_results = []

for representation_mode in ["lexicalized", "morphosyntax_only", "mixed"]:
    experiment_result = train_and_select_representation(
        representation_mode=representation_mode,
        train_example_list=english_train_for_comparison,
        dev_example_list=english_dev_for_comparison,
        test_example_list=english_test_for_comparison,
        batch_size=32,
        epoch_count=8,
        learning_rate=0.003,
    )
    english_results.append(experiment_result)

lexicalized epoch=1 train_total_loss=6.3686 dev_uas=0.1371 dev_las=0.0776
lexicalized epoch=2 train_total_loss=4.9234 dev_uas=0.1772 dev_las=0.0933
lexicalized epoch=3 train_total_loss=4.4453 dev_uas=0.1939 dev_las=0.1121
lexicalized epoch=4 train_total_loss=4.0788 dev_uas=0.2114 dev_las=0.1278
lexicalized epoch=5 train_total_loss=3.7944 dev_uas=0.2203 dev_las=0.1388
lexicalized epoch=6 train_total_loss=3.5525 dev_uas=0.2359 dev_las=0.1538
lexicalized epoch=7 train_total_loss=3.3775 dev_uas=0.2376 dev_las=0.1615
lexicalized epoch=8 train_total_loss=3.2241 dev_uas=0.2355 dev_las=0.1618
morphosyntax_only epoch=1 train_total_loss=4.1563 dev_uas=0.3212 dev_las=0.2540
morphosyntax_only epoch=2 train_total_loss=2.8768 dev_uas=0.3381 dev_las=0.2726
morphosyntax_only epoch=3 train_total_loss=2.7786 dev_uas=0.3295 dev_las=0.2721
morphosyntax_only epoch=4 train_total_loss=2.7354 dev_uas=0.3399 dev_las=0.2842
morphosyntax_only epoch=5 train_total_loss=2.7103 dev_uas=0.3450 dev_las=0.2897
morphosy

In [58]:
for experiment_result in english_results:
    best_dev_result = experiment_result["best_dev_result"]
    test_metrics = experiment_result["test_metrics"]
    print(
        f"{experiment_result['representation_mode']}: "
        f"best_dev_epoch={best_dev_result['epoch']} "
        f"best_dev_uas={best_dev_result['dev_uas']:.4f} "
        f"best_dev_las={best_dev_result['dev_las']:.4f} "
        f"test_uas={test_metrics['UAS']:.4f} "
        f"test_las={test_metrics['LAS']:.4f}"
    )

lexicalized: best_dev_epoch=8 best_dev_uas=0.2355 best_dev_las=0.1618 test_uas=0.2542 test_las=0.1770
morphosyntax_only: best_dev_epoch=5 best_dev_uas=0.3450 best_dev_las=0.2897 test_uas=0.3409 test_las=0.2871
mixed: best_dev_epoch=8 best_dev_uas=0.3297 best_dev_las=0.2812 test_uas=0.3398 test_las=0.2949


### Brief Discussion of Section 6B (English Test Comparison)

On English test data, the mixed setting performed best with LAS 0.2949, while morphosyntax-only was very close at 0.2871 and lexicalized was again much lower at 0.1770.

So morphosyntax-only still clearly beats lexicalized in English, but the gap is smaller than it was in Czech.

Unlike Czech, English gets a small extra benefit from mixing lexical and morphosyntactic input.

This suggests that English also benefits from morphosyntax, but lexical identity matters a bit more here than it did in Czech.

In [59]:
# summary table to test UAS/LAS for all settings and both languages
czech_best_by_mode = {
    result["representation_mode"]: result
    for result in czech_results
}

english_best_by_mode = {
    result["representation_mode"]: result
    for result in english_results
}

comparison_rows = [
    {
        "language": "Czech",
        "representation": "lexicalized",
        "test_uas": czech_best_by_mode["lexicalized"]["test_metrics"]["UAS"],
        "test_las": czech_best_by_mode["lexicalized"]["test_metrics"]["LAS"],
    },
    {
        "language": "Czech",
        "representation": "morphosyntax_only",
        "test_uas": czech_best_by_mode["morphosyntax_only"]["test_metrics"]["UAS"],
        "test_las": czech_best_by_mode["morphosyntax_only"]["test_metrics"]["LAS"],
    },
    {
        "language": "Czech",
        "representation": "mixed",
        "test_uas": czech_best_by_mode["mixed"]["test_metrics"]["UAS"],
        "test_las": czech_best_by_mode["mixed"]["test_metrics"]["LAS"],
    },
    {
        "language": "English",
        "representation": "lexicalized",
        "test_uas": english_best_by_mode["lexicalized"]["test_metrics"]["UAS"],
        "test_las": english_best_by_mode["lexicalized"]["test_metrics"]["LAS"],
    },
    {
        "language": "English",
        "representation": "morphosyntax_only",
        "test_uas": english_best_by_mode["morphosyntax_only"]["test_metrics"]["UAS"],
        "test_las": english_best_by_mode["morphosyntax_only"]["test_metrics"]["LAS"],
    },
    {
        "language": "English",
        "representation": "mixed",
        "test_uas": english_best_by_mode["mixed"]["test_metrics"]["UAS"],
        "test_las": english_best_by_mode["mixed"]["test_metrics"]["LAS"],
    },
]

for row in comparison_rows:
    print(
        f"{row['language']:7s} "
        f"{row['representation']:18s} "
        f"UAS={row['test_uas']:.4f} "
        f"LAS={row['test_las']:.4f}"
    )


Czech   lexicalized        UAS=0.2189 LAS=0.1318
Czech   morphosyntax_only  UAS=0.4382 LAS=0.4017
Czech   mixed              UAS=0.4212 LAS=0.3749
English lexicalized        UAS=0.2542 LAS=0.1770
English morphosyntax_only  UAS=0.3409 LAS=0.2871
English mixed              UAS=0.3398 LAS=0.2949


In [60]:
# delta LAS between morphosyntax-only and lexicalized
czech_delta_las_morph_minus_lex = (
    czech_best_by_mode["morphosyntax_only"]["test_metrics"]["LAS"]
    - czech_best_by_mode["lexicalized"]["test_metrics"]["LAS"]
)

english_delta_las_morph_minus_lex = (
    english_best_by_mode["morphosyntax_only"]["test_metrics"]["LAS"]
    - english_best_by_mode["lexicalized"]["test_metrics"]["LAS"]
)

print(f"Czech delta LAS (morphosyntax_only - lexicalized): {czech_delta_las_morph_minus_lex:.4f}")
print(f"English delta LAS (morphosyntax_only - lexicalized): {english_delta_las_morph_minus_lex:.4f}")

Czech delta LAS (morphosyntax_only - lexicalized): 0.2699
English delta LAS (morphosyntax_only - lexicalized): 0.1102


### Brief Discussion of Section 6C (Cross-Language Comparison)

The main cross-language result is the LAS gain of morphosyntax-only over lexicalized, which is 0.2699 in Czech but only 0.1102 in English.

Morphosyntax-only also reaches a higher absolute test LAS in Czech (0.4017) than in English (0.2871).

This supports the hypothesis that overt morphosyntactic marking carries stronger syntactic information in Czech than in English.

Because the parser is simple and the data is subset-based, I treat this as controlled evidence rather than a final claim about the languages in general.

## Section 6D: Label Confusion Analysis

To make the comparison more informative, I now inspect label-wise confusion patterns (e.g., `nsubj` vs. `obj`) to understand which dependency relations depend on morphosyntax versus lexical identity.

I focus on the lexicalized and morphosyntax-only models because that is the main contrast.

In [61]:
from collections import Counter

def collect_mst_token_records(
    example_list,
    batch_size,
    token_encoder,
    arc_scorer,
    label_scorer,
    ignore_punct=True,
):
    """Collect gold and predicted information for each evaluated token."""
    token_encoder.eval()
    arc_scorer.eval()
    label_scorer.eval()

    punct_upos_id = upos_to_id.get("PUNCT", None)
    record_list = []
    batch_list = create_batches(example_list, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch in batch_list:
            prediction_dict = predict_batch_with_mst(
                batch,
                token_encoder,
                arc_scorer,
                label_scorer,
            )

            batch_size_now, sequence_length = batch["head_indices"].shape

            for batch_index in range(batch_size_now):
                for dependent_index in range(1, sequence_length):
                    if batch["head_indices"][batch_index, dependent_index].item() == ignore_index:
                        continue

                    if ignore_punct and punct_upos_id is not None:
                        if batch["upos_ids"][batch_index, dependent_index].item() == punct_upos_id:
                            continue

                    gold_head = batch["head_indices"][batch_index, dependent_index].item()
                    pred_head = prediction_dict["predicted_head_indices"][batch_index, dependent_index].item()
                    gold_label_id = batch["label_ids"][batch_index, dependent_index].item()
                    pred_label_id = prediction_dict["predicted_label_ids"][batch_index, dependent_index].item()

                    record_list.append(
                        {
                            "gold_head": gold_head,
                            "pred_head": pred_head,
                            "gold_label": id_to_label[gold_label_id],
                            "pred_label": id_to_label[pred_label_id],
                            "distance": abs(dependent_index - gold_head),
                        }
                    )

    return record_list

def print_top_label_confusions(record_list, title, top_n=8):
    """Print the most common label confusions when the head is correct."""
    confusion_counter = Counter()

    for record in record_list:
        if record["gold_head"] != record["pred_head"]:
            continue
        if record["gold_label"] == record["pred_label"]:
            continue
        confusion_counter[(record["gold_label"], record["pred_label"])] += 1

    print(title)
    if not confusion_counter:
        print("  No label confusions with correct heads.")
        return

    for (gold_label, pred_label), count in confusion_counter.most_common(top_n):
        print(f"  gold={gold_label:12s} pred={pred_label:12s} count={count}")

In [62]:
# collect per token records for the two main settings in each language
analysis_records = {
    "Czech lexicalized": collect_mst_token_records(
        czech_test_for_comparison,
        batch_size=32,
        token_encoder=czech_best_by_mode["lexicalized"]["token_encoder"],
        arc_scorer=czech_best_by_mode["lexicalized"]["arc_scorer"],
        label_scorer=czech_best_by_mode["lexicalized"]["label_scorer"],
    ),
    "Czech morphosyntax_only": collect_mst_token_records(
        czech_test_for_comparison,
        batch_size=32,
        token_encoder=czech_best_by_mode["morphosyntax_only"]["token_encoder"],
        arc_scorer=czech_best_by_mode["morphosyntax_only"]["arc_scorer"],
        label_scorer=czech_best_by_mode["morphosyntax_only"]["label_scorer"],
    ),
    "English lexicalized": collect_mst_token_records(
        english_test_for_comparison,
        batch_size=32,
        token_encoder=english_best_by_mode["lexicalized"]["token_encoder"],
        arc_scorer=english_best_by_mode["lexicalized"]["arc_scorer"],
        label_scorer=english_best_by_mode["lexicalized"]["label_scorer"],
    ),
    "English morphosyntax_only": collect_mst_token_records(
        english_test_for_comparison,
        batch_size=32,
        token_encoder=english_best_by_mode["morphosyntax_only"]["token_encoder"],
        arc_scorer=english_best_by_mode["morphosyntax_only"]["arc_scorer"],
        label_scorer=english_best_by_mode["morphosyntax_only"]["label_scorer"],
    ),
}

for name, record_list in analysis_records.items():
    print_top_label_confusions(record_list, title=name)
    print()

Czech lexicalized
  gold=nmod         pred=amod         count=59
  gold=conj         pred=amod         count=35
  gold=obj          pred=amod         count=22
  gold=flat         pred=amod         count=14
  gold=nsubj        pred=amod         count=14
  gold=obl          pred=amod         count=13
  gold=obl          pred=nmod         count=11
  gold=obl:arg      pred=obl          count=11

Czech morphosyntax_only
  gold=obl          pred=obj          count=23
  gold=obl:arg      pred=obl          count=18
  gold=advmod:emph  pred=advmod       count=17
  gold=obl          pred=obl:arg      count=12
  gold=obl:arg      pred=obj          count=11
  gold=conj         pred=nmod         count=11
  gold=expl:pass    pred=expl:pv      count=10
  gold=nsubj:pass   pred=nsubj        count=8

English lexicalized
  gold=flat         pred=amod         count=50
  gold=obj          pred=nsubj        count=25
  gold=nsubj        pred=obl          count=23
  gold=obj          pred=obl          count=

### Brief Discussion of Section 6D (Label Confusions)

The label confusion lists are informative because they show what goes wrong beyond the overall LAS numbers.

For Czech, the lexicalized model collapses many labels into `amod`, while the morphosyntax-only model makes more specific mistakes such as `obl` vs `obj` or `obl:arg`.

For English, even the morphosyntax-only model still often confuses `obl`, `obj`, and `nsubj`, which fits the weaker surface morphology of English.

These patterns help explain why morphosyntax-only gives a much larger boost in Czech than in English.

## Section 6E: Accuracy by Dependency Length

Checking whether short dependencies are easier than long ones.

I again compare lexicalized and morphosyntax-only on the held-out test data.

In [63]:
def length_bucket(distance):
    """Map one dependency length to a small analysis bin."""
    if distance <= 1:
        return "1"
    if distance == 2:
        return "2"
    if distance <= 4:
        return "3-4"
    if distance <= 6:
        return "5-6"
    return "7+"

def print_length_scores(record_list, title):
    """Print UAS and LAS for a few dependency-length bins."""
    bucket_totals = Counter()
    bucket_uas_correct = Counter()
    bucket_las_correct = Counter()

    for record in record_list:
        bucket = length_bucket(record["distance"])
        bucket_totals[bucket] += 1

        if record["gold_head"] == record["pred_head"]:
            bucket_uas_correct[bucket] += 1
            if record["gold_label"] == record["pred_label"]:
                bucket_las_correct[bucket] += 1

    print(title)
    for bucket in ["1", "2", "3-4", "5-6", "7+"]:
        total = bucket_totals[bucket]
        if total == 0:
            continue
        uas = bucket_uas_correct[bucket] / total
        las = bucket_las_correct[bucket] / total
        print(f"  distance={bucket:3s} total={total:4d} UAS={uas:.4f} LAS={las:.4f}")

In [64]:
for name, record_list in analysis_records.items():
    print_length_scores(record_list, title=name)
    print()

Czech lexicalized
  distance=1   total=2595 UAS=0.2555 LAS=0.1788
  distance=2   total=1337 UAS=0.1960 LAS=0.0987
  distance=3-4 total=1023 UAS=0.2023 LAS=0.1036
  distance=5-6 total= 389 UAS=0.2005 LAS=0.0925
  distance=7+  total= 439 UAS=0.1276 LAS=0.0547

Czech morphosyntax_only
  distance=1   total=2595 UAS=0.5329 LAS=0.5118
  distance=2   total=1337 UAS=0.3777 LAS=0.3426
  distance=3-4 total=1023 UAS=0.3842 LAS=0.3177
  distance=5-6 total= 389 UAS=0.3445 LAS=0.2879
  distance=7+  total= 439 UAS=0.2711 LAS=0.2278

English lexicalized
  distance=1   total=2426 UAS=0.2683 LAS=0.2040
  distance=2   total=1566 UAS=0.2554 LAS=0.1782
  distance=3-4 total=1377 UAS=0.2542 LAS=0.1721
  distance=5-6 total= 423 UAS=0.2553 LAS=0.1466
  distance=7+  total= 526 UAS=0.1844 LAS=0.0856

English morphosyntax_only
  distance=1   total=2426 UAS=0.4114 LAS=0.3817
  distance=2   total=1566 UAS=0.3391 LAS=0.2886
  distance=3-4 total=1377 UAS=0.2861 LAS=0.2244
  distance=5-6 total= 423 UAS=0.2648 LAS=0.16

### Brief Discussion of Section 6E (Dependency Length)

In both languages, short dependencies are easier than long ones, which is what we would expect in dependency parsing.

However, morphosyntax-only beats lexicalized in every distance bin, so the improvement is not limited to the easiest local attachments.

The Czech advantage stays especially clear even for longer dependencies, while the English gains become smaller at longer distances.

This again supports the idea that Czech morphosyntactic cues remain useful across a wider range of syntactic configurations.

## End of Notebook

The detailed discussion of results, limitations, and future directions is in the accompanying project report.